# DP2 DNF cModel photo-z QA

This report compares the `griz` and `riz` catalogs. Catalog statistics and redshift distributions use all HATS rows. The accuracy section uses a common spectroscopic reference sample held out from both models.

In [ ]:
import atexit
from datetime import datetime, timezone
from html import escape
import json
import os
from pathlib import Path
import sys

from dask import delayed
from IPython.display import HTML, Markdown, display
import lsdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

VARIANTS = ("griz", "riz")
ESTIMATORS = ("dnf_z_mean", "dnf_z_best", "DNF_ZN")
HIST_COLUMNS = ("dnf_z_best", "DNF_ZN")
HIST_EDGES = np.linspace(0.0, 6.0, 121)
FLAG_SELECTIONS = {"Flag 4": (4,), "Flags 3 and 4": (3, 4)}

dp2_root = Path(os.environ["PZ_DP2_ROOT"])
run_root = Path(os.environ["PZ_DP2_CUSTOM_ROOT"])
repo = Path(os.environ["PZ_COMPUTE_DIR"])
reference_path = Path(os.environ.get(
    "PZ_QA_TEST_SET", dp2_root / "common/validation/dp2_clean_test_set.parquet"
))
output_dir = Path(os.environ.get("PZ_QA_OUTPUT_DIR", run_root / "qa/delivery"))

sys.path.insert(0, str(repo / "doc/notebooks/pz_compute_v2"))
from pz_qa import start_cluster

plt.style.use("seaborn-v0_8-whitegrid")

## Spectroscopic reference

The input contains spectroscopic matches after excluding objects used to train both models. We compare two redshift-quality selections, flag 4 alone and flags 3 and 4 together. Within each selection, each object contributes once. For repeated measurements, we keep the highest available flag and take the median of measurements at that flag if their full spread divided by `1 + median redshift` is at most 0.001. We omit objects with conflicting measurements at the highest flag.

In [ ]:
def reference_objects(source, flags):
    selected = source.loc[source["z_flag_homogenized"].isin(flags)]
    valid = selected.loc[
        selected["objectId"].notna()
        & np.isfinite(selected["redshift"])
        & (selected["redshift"] >= 0)
    ]
    best_flag = valid.groupby("objectId")["z_flag_homogenized"].transform("max")
    grouped = valid.loc[valid["z_flag_homogenized"] == best_flag].groupby("objectId")
    z = grouped["redshift"].agg(["min", "max", "median"])
    agrees = (z["max"] - z["min"]) <= 0.001 * (1 + z["median"])
    return z.loc[agrees, ["median"]].rename(columns={"median": "z_spec"}), {
        "reference rows": len(selected),
        "usable reference rows": len(valid),
        "conflicting objects excluded": int((~agrees).sum()),
        "reference objects": int(agrees.sum()),
    }


reference = pd.read_parquet(reference_path, columns=[
    "objectId", "redshift", "instrument_type_homogenized", "z_flag_homogenized",
])
reference = reference.loc[reference["instrument_type_homogenized"].eq("s")]
samples = {label: reference_objects(reference, flags) for label, flags in FLAG_SELECTIONS.items()}
reference_ids = np.unique(np.concatenate([sample.index.to_numpy() for sample, _ in samples.values()]))
display(pd.DataFrame({label: counts for label, (_, counts) in samples.items()}).T)

In [ ]:
CLUSTER_TYPE = os.environ.get("PZ_QA_DASK_CLUSTER", "slurm")
if CLUSTER_TYPE not in ("slurm", "local"):
    raise ValueError("PZ_QA_DASK_CLUSTER must be 'slurm' or 'local'")
CLUSTER_SETTINGS = {
    "type": CLUSTER_TYPE,
    "minimum_jobs": int(os.environ.get("PZ_QA_DASK_MINIMUM_JOBS", "16")),
    "maximum_jobs": int(os.environ.get("PZ_QA_DASK_MAXIMUM_JOBS", "48")),
    "cores": int(os.environ.get("PZ_QA_DASK_CORES", "2")),
    "memory": os.environ.get("PZ_QA_DASK_MEMORY", "40GiB"),
    "processes": 1 if CLUSTER_TYPE == "slurm" else False,
    "queue": os.environ.get("PZ_QA_DASK_QUEUE", "cpu_photoz"),
    "account": os.environ.get("PZ_QA_DASK_ACCOUNT", "hpc-photoz"),
    "interface": os.environ.get("PZ_QA_DASK_INTERFACE") or None,
    "walltime": os.environ.get("PZ_QA_DASK_WALLTIME", "01:00:00"),
    "wait_timeout": int(os.environ.get("PZ_QA_DASK_WAIT_TIMEOUT", "600")),
    "job_extra_directives": ["--propagate"],
    "n_workers": int(os.environ.get("PZ_QA_DASK_LOCAL_WORKERS", "2")),
    "threads_per_worker": int(os.environ.get("PZ_QA_DASK_LOCAL_THREADS", "1")),
    "memory_limit": os.environ.get("PZ_QA_DASK_LOCAL_MEMORY", "4GiB"),
}
if CLUSTER_SETTINGS["minimum_jobs"] > CLUSTER_SETTINGS["maximum_jobs"]:
    raise ValueError("PZ_QA_DASK_MINIMUM_JOBS must not exceed PZ_QA_DASK_MAXIMUM_JOBS")
client, cluster = start_cluster({"cluster": CLUSTER_SETTINGS})

def close_dask_resources():
    client.close()
    if cluster is not None:
        cluster.close()

atexit.register(close_dask_resources)

## Catalog sizes, columns and descriptive statistics

Counts, means, sample standard deviations, and extrema are computed across all main-catalog partitions using finite values. For `objectId`, only counts and extrema are reported.

In [ ]:
def scan_partition(frame, ids):
    stats = {}
    for name in frame:
        if name == "objectId":
            values = frame[name].dropna()
            stats[name] = (len(values), values.min(), values.max(), None, None)
            continue
        values = frame[name].to_numpy(dtype=np.float64, na_value=np.nan)
        finite = values[np.isfinite(values)]
        if finite.size:
            mean = finite.mean()
            m2 = np.square(finite - mean).sum()
            stats[name] = (len(finite), finite.min(), finite.max(), mean, m2)
        else:
            stats[name] = (0, None, None, None, None)

    histograms = {
        name: np.histogram(frame[name].to_numpy(dtype=np.float64, na_value=np.nan), HIST_EDGES)[0]
        for name in HIST_COLUMNS
    }
    matched = frame.loc[frame["objectId"].isin(ids), [
        "objectId", *ESTIMATORS,
        "dnf_z_err68_low", "dnf_z_err68_high", "dnf_z_err95_low", "dnf_z_err95_high",
    ]]
    return len(frame), stats, histograms, matched


def global_scan(catalog, ids):
    parts = catalog.to_dask_dataframe().to_delayed()
    results = client.gather(client.compute([delayed(scan_partition)(part, ids) for part in parts]))
    if not results:
        raise ValueError("The HATS catalog contains no partitions.")
    rows = sum(part[0] for part in results)
    stats = []
    for name in catalog.columns:
        records = [part[1][name] for part in results if part[1][name][0]]
        count = sum(record[0] for record in records)
        row = {"column": name, "finite": count, "missing/non-finite": rows - count}
        if count:
            row["min"] = min(record[1] for record in records)
            row["max"] = max(record[2] for record in records)
        if name != "objectId" and count:
            mean = sum(n * avg for n, _, _, avg, _ in records) / count
            variance_sum = sum(m2 + n * (avg - mean)**2 for n, _, _, avg, m2 in records)
            row["mean"] = mean
            row["std"] = np.sqrt(variance_sum / (count - 1)) if count > 1 else None
        stats.append(row)
    histograms = {name: np.sum([part[2][name] for part in results], axis=0) for name in HIST_COLUMNS}
    matched = pd.concat([part[3] for part in results], ignore_index=True)
    return rows, stats, histograms, matched


catalogs = {
    variant: lsdb.open_catalog(
        run_root / variant / "stage3/output" / f"pz-dnf-cmodel-{variant}-scalar-hats",
        columns="all",
    )
    for variant in VARIANTS
}
scans = {variant: global_scan(catalog, reference_ids) for variant, catalog in catalogs.items()}
display(pd.DataFrame([
    {"bands": variant, "rows": scans[variant][0], "columns": len(catalog.columns)}
    for variant, catalog in catalogs.items()
]).set_index("bands"))
for variant, catalog in catalogs.items():
    items = "".join(f"<li><code>{escape(name)}</code></li>" for name in catalog.columns)
    display(Markdown(f"### {variant}"))
    display(HTML(
        '<div style="max-height:8em; overflow-y:auto; border:1px solid var(--jp-border-color2, #888)">'
        f"<ol>{items}</ol></div>"
    ))
    stats = pd.DataFrame(scans[variant][1], dtype=object).set_index("column")
    display(stats)

## Global redshift distributions

Both distributions count all finite estimates in the displayed range `[0, 6]`. The table reports the finite estimates outside the plotted range separately.

In [ ]:
histogram_counts = []
for name in HIST_COLUMNS:
    fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
    for variant, (_, stats, histograms, _) in scans.items():
        counts = histograms[name]
        finite = next(row["finite"] for row in stats if row["column"] == name)
        ax.stairs(counts, HIST_EDGES, label=variant)
        histogram_counts.append({
            "estimate": name, "bands": variant, "plotted": int(counts.sum()),
            "finite outside [0, 6]": int(finite - counts.sum()),
        })
    ax.set(xlabel=name, ylabel="Objects per bin")
    ax.legend()
    plt.show()
display(pd.DataFrame(histogram_counts))

## Photo-z versus spectroscopy

For each flag selection and estimator, both band variants are compared on the same reference objects with finite estimates in both catalogs. The availability table shows catalog matches and finite estimates before that pairing. We report the median of `Δz = (z_phot - z_spec) / (1 + z_spec)`, NMAD (1.4826 times the median absolute deviation about the median), and the fraction with `|Δz| > 0.15`. Interval coverage is the fraction of spectra within the published 68% and 95% bounds on the common sample with finite bounds in both variants.

Each scatter plot uses the identity line. The plotted range includes all paired points. These metrics describe the spectroscopic selection and its coverage in the two catalogs.

In [ ]:
metrics = []
availability = []
for label, (sample, _) in samples.items():
    paired = sample.copy()
    for variant, (_, _, _, matched) in scans.items():
        photo = matched.set_index("objectId").add_suffix(f"_{variant}")
        paired = paired.join(photo, validate="one_to_one")
        for estimator in ESTIMATORS:
            availability.append({
                "selection": label, "bands": variant, "estimate": estimator,
                "reference objects": len(sample),
                "matched in HATS": int(sample.index.isin(photo.index).sum()),
                "finite estimates": int(np.isfinite(paired[f"{estimator}_{variant}"]).sum()),
            })

    fig, axes = plt.subplots(len(ESTIMATORS), 2, figsize=(9, 12), layout="constrained")
    fig.suptitle(label)
    for row_axes, estimator in zip(axes, ESTIMATORS):
        phot_cols = [f"{estimator}_{variant}" for variant in VARIANTS]
        common = paired.loc[np.isfinite(paired[phot_cols]).all(axis=1)]
        values = common[["z_spec", *phot_cols]].to_numpy(dtype=float)
        limit = (min(0, values.min()), max(0.1, values.max())) if len(values) else (0, 6)
        for ax, variant, col in zip(row_axes, VARIANTS, phot_cols):
            ax.scatter(common["z_spec"], common[col], s=4, alpha=0.3, rasterized=True)
            ax.plot(limit, limit, color="0.5", linewidth=1)
            ax.set(xlim=limit, ylim=limit, xlabel="Spectroscopic redshift",
                   ylabel=estimator, title=f"{variant} (N={len(common):,})")
            ax.set_aspect("equal", adjustable="box")
            dz = (common[col] - common["z_spec"]) / (1 + common["z_spec"])
            bias = dz.median() if len(dz) else np.nan
            metrics.append({
                "selection": label, "estimate": estimator, "bands": variant,
                "paired objects": len(dz), "median bias": bias,
                "NMAD": 1.4826 * (dz - bias).abs().median() if len(dz) else np.nan,
                "outlier fraction": (dz.abs() > 0.15).mean(),
            })
    plt.show()

display(pd.DataFrame(availability))
display(pd.DataFrame(metrics))

coverage = []
for label, (sample, _) in samples.items():
    joined = sample.copy()
    for variant, (_, _, _, matched) in scans.items():
        joined = joined.join(matched.set_index("objectId").add_suffix(f"_{variant}"), validate="one_to_one")
    for level in (68, 95):
        bounds = [(f"dnf_z_err{level}_low_{v}", f"dnf_z_err{level}_high_{v}") for v in VARIANTS]
        valid = joined.loc[np.isfinite(joined[[col for pair in bounds for col in pair]]).all(axis=1)]
        for variant, (low, high) in zip(VARIANTS, bounds):
            coverage.append({
                "selection": label, "interval": level, "bands": variant,
                "paired objects": len(valid),
                "coverage": valid["z_spec"].between(valid[low], valid[high]).mean(),
            })
display(pd.DataFrame(coverage))

In [ ]:
report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "histogram_edges": HIST_EDGES.tolist(),
    "catalogs": {
        variant: {
            "rows": rows, "columns": len(stats), "describe": stats,
            "histograms": {key: values.tolist() for key, values in histograms.items()},
        }
        for variant, (rows, stats, histograms, _) in scans.items()
    },
    "reference_selections": {label: counts for label, (_, counts) in samples.items()},
    "availability": availability,
    "accuracy": json.loads(pd.DataFrame(metrics).to_json(orient="records")),
    "coverage": json.loads(pd.DataFrame(coverage).to_json(orient="records")),
}
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "qa_summary.json").write_text(json.dumps(report, indent=2, allow_nan=False, default=lambda x: x.item()) + "\n")
atexit.unregister(close_dask_resources)
close_dask_resources()